In [1]:
!pip install pyspark

In [3]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()


In [15]:
from google.colab import files,output
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
display(df.head())

Saving manufacturing_production_120k.csv to manufacturing_production_120k (1).csv


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [16]:
from IPython.display import display
from google.colab import output
import ipywidgets as widgets

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)
# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [17]:
from google.colab import data_table
data_table.enable_dataframe_formatter()
display(df)

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,PR0119996,2026-07-13,PL05,MC022,OP0193,Evening,MTR-200,Motors,435,399,1,73.0,126.07,62.79,5.73,Rework,No,Bharat
119996,PR0119997,2026-04-26,PL05,MC024,OP0124,Evening,CTL-130,Controllers,813,777,8,49.0,318.22,63.05,3.53,Pass,No,Irfan
119997,PR0119998,2026-02-01,PL06,MC027,OP0051,Evening,VLV-120,Valves,650,597,13,0.0,269.46,60.52,3.81,Pass,No,Bharat
119998,PR0119999,2026-01-17,PL01,MC001,OP0452,Evening,MTR-300,Motors,887,899,22,21.0,184.62,64.89,5.20,Pass,Yes,Farah


In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [19]:
filtered_df = df.filter(col("quantity")>3)
print("Transformation created")
selected_df = filtered_df.select("city", "product", "quantity")
print("Another transformation created")
selected_df.show()

Transformation created
Another transformation created
+---------+-------+--------+
|     city|product|quantity|
+---------+-------+--------+
|Bangalore| Mobile|       5|
|Hyderabad| Mobile|       4|
|Bangalore| Tablet|       6|
|   Mumbai| Mobile|       8|
|Hyderabad| Tablet|       5|
|   Mumbai| Tablet|       7|
|  Chennai| Mobile|       6|
+---------+-------+--------+



In [20]:
from google.colab import files
uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [21]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("LazyExecutionExercises").getOrCreate()

In [22]:
sales_df = spark.read.option("header", True).option("inferSchema", True).csv("/content/lazy_execution_sales_25000.csv")
products_df = spark.read.option("header", True).option("inferSchema", True).csv("/content/lazy_execution_products.csv")
stores_df = spark.read.option("header", True).option("inferSchema", True).csv("/content/lazy_execution_stores.csv")
print("Sales:", sales_df.count(), "Products:", products_df.count(), "Stores:", stores_df.count())
sales_df.printSchema()

Sales: 25000 Products: 15 Stores: 8
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- sales_channel: string (nullable = true)



Part A


In [23]:
quantity_over_5_df = sales_df.filter(F.col("quantity") > 5)
quantity_over_5_df.show()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [24]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show(20, truncate=False)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|O000001 |P002      |3       |2999      |
|O000002 |P014      |6       |29999     |
|O000003 |P008      |6       |22999     |
|O000004 |P008      |7       |22999     |
|O000005 |P007      |2       |79999     |
|O000006 |P005      |8       |2499      |
|O000007 |P010      |8       |3999      |
|O000008 |P002      |9       |2999      |
|O000009 |P007      |5       |79999     |
|O000010 |P006      |5       |55999     |
|O000011 |P007      |1       |79999     |
|O000012 |P010      |9       |3999      |
|O000013 |P013      |6       |5999      |
|O000014 |P005      |8       |2499      |
|O000015 |P007      |7       |79999     |
|O000016 |P006      |1       |55999     |
|O000017 |P004      |9       |16999     |
|O000018 |P012      |7       |6999      |
|O000019 |P014      |5       |29999     |
|O000020 |P010      |7       |3999      |
+--------+----------+--------+----

In [25]:
expensive_df = sales_df.filter(F.col("unit_price") > 20000)
print("Matching records:", expensive_df.count())

Matching records: 6560


In [26]:
gross_df = sales_df.withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
gross_df.explain()
gross_df.show(20, truncate=False)

== Physical Plan ==
*(1) Project [order_id#48, order_date#49, store_id#50, product_id#51, quantity#52, unit_price#53, discount_pct#54, payment_method#55, order_status#56, sales_channel#57, (quantity#52 * unit_price#53) AS gross_amount#206]
+- FileScan csv [order_id#48,order_date#49,store_id#50,product_id#51,quantity#52,unit_price#53,discount_pct#54,payment_method#55,order_status#56,sales_channel#57] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------+----------+--------+-

In [27]:
completed_sorted_df = sales_df.filter(F.col("order_status") == "Completed").orderBy(F.col("unit_price").desc())
for row in completed_sorted_df.take(10):
    print(row)

Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online')
Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online')
Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, unit_pr

Part B

In [28]:
completed_revenue_df = (sales_df.filter(F.col("order_status") == "Completed")
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
    .select("order_id", "product_id", "quantity", "unit_price", "gross_amount")
    .orderBy(F.col("gross_amount").desc()))
completed_revenue_df.show(20, truncate=False)

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
|O002603 |P007      |9       |79999     |719991      |
|O006170 |P007      |9       |79999     |719991      |
|O002693 |P007      |9       |79999     |719991      |
|O000107 |P007      |9       |79999     |719991      |
|O002841 |P007      |9       |79999     |719991      |
|O000367 |P007      |9       |79999     |719991      |
|O002940 |P007      |9       |79999     |719991      |
|O000527 |P007      |9       |79999     |719991      |
|O003108 |P007      |9       |79999     |719991      |
|O000690 |P007      |9       |79999     |719991      |
|O003542 |P007      |9       |79999     |719991      |
|O001201 |P007      |9       |79999     |719991      |
|O003884 |P007      |9       |79999     |719991      |
|O001300 |P007      |9       |79999     |719991      |
|O003897 |P007      |9       |79999     |719991      |
|O001796 |

In [29]:
discounted_orders_df = (sales_df.filter(F.col("discount_pct") >= 10)
    .withColumn("discounted_sales_amount", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100.0))
    .orderBy(F.col("discounted_sales_amount").desc()))
discounted_orders_df.show(15, truncate=False)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|O006653 |2026-06-29|S01     |P007      |9       |79999     |10          |Card          |Cancelled   |Online       |647991.9               |
|O011893 |2026-03-26|S07     |P007      |9       |79999     |10          |UPI           |Completed   |Online       |647991.9               |
|O006699 |2026-06-25|S05     |P007      |9       |79999     |10          |Net Banking   |Completed   |Online       |647991.9               |
|O003108 |2026-02-23|S08     |P007      |9       |79999     |10          |Net Banking   |Completed   |In Store     |647991.9               |
|O007743 |202

In [30]:
sales_df.groupBy("payment_method").agg(F.sum("quantity").alias("total_quantity")).orderBy(F.col("total_quantity").desc()).show(truncate=False)

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|Cash          |31754         |
|Card          |31327         |
|UPI           |30894         |
|Net Banking   |30889         |
+--------------+--------------+



In [31]:
sales_df.groupBy("sales_channel", "order_status").agg(F.count("*").alias("transaction_count")).show(truncate=False)

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|Online       |Completed   |10797            |
|In Store     |Cancelled   |592              |
|Online       |Returned    |987              |
|Online       |Cancelled   |633              |
|In Store     |Returned    |1013             |
|In Store     |Completed   |10978            |
+-------------+------------+-----------------+



In [32]:
reduced_df = sales_df.select("order_id", "store_id", "product_id", "quantity").drop("store_id")
print(reduced_df.first())

Row(order_id='O000001', product_id='P002', quantity=3)


Part C

In [33]:
sales_df.join(products_df, "product_id", "inner").select("product_name", "category", "quantity", "unit_price").show(20, truncate=False)

+-------------------+-----------+--------+----------+
|product_name       |category   |quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|3       |2999      |
|Smartphone 5G      |Mobiles    |6       |29999     |
|Android Tablet     |Tablets    |6       |22999     |
|Android Tablet     |Tablets    |7       |22999     |
|Laptop 16          |Computers  |2       |79999     |
|Webcam HD          |Accessories|8       |2499      |
|Bluetooth Speaker  |Audio      |8       |3999      |
|Mechanical Keyboard|Accessories|9       |2999      |
|Laptop 16          |Computers  |5       |79999     |
|Laptop 14          |Computers  |5       |55999     |
|Laptop 16          |Computers  |1       |79999     |
|Bluetooth Speaker  |Audio      |9       |3999      |
|External HDD 2TB   |Storage    |6       |5999      |
|Webcam HD          |Accessories|8       |2499      |
|Laptop 16          |Computers  |7       |79999     |
|Laptop 14          |Compute

In [34]:
sales_df.join(stores_df, "store_id", "inner").groupBy("city").agg(F.sum("quantity").alias("total_quantity_sold")).orderBy(F.col("total_quantity_sold").desc()).show(truncate=False)

+---------+-------------------+
|city     |total_quantity_sold|
+---------+-------------------+
|Jaipur   |16019              |
|Hyderabad|15784              |
|Chennai  |15727              |
|Bengaluru|15718              |
|Kolkata  |15619              |
|Mumbai   |15489              |
|Pune     |15407              |
|Delhi    |15101              |
+---------+-------------------+



In [35]:
category_region_revenue_df = (sales_df.filter(F.col("order_status") == "Completed")
    .join(products_df, "product_id", "inner")
    .join(stores_df, "store_id", "inner")
    .withColumn("revenue", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100.0))
    .groupBy("category", "region")
    .agg(F.round(F.sum("revenue"), 2).alias("total_completed_revenue"))
    .orderBy(F.col("total_completed_revenue").desc()))
category_region_revenue_df.show(truncate=False)

+-----------+------+-----------------------+
|category   |region|total_completed_revenue|
+-----------+------+-----------------------+
|Computers  |South |3.5266921295E8         |
|Computers  |North |2.273618241E8          |
|Computers  |West  |2.0921530285E8         |
|Computers  |East  |1.1648428645E8         |
|Mobiles    |South |1.144250127E8          |
|Mobiles    |West  |7.639817E7             |
|Mobiles    |North |7.14025309E7           |
|Tablets    |South |6.050691915E7          |
|Displays   |South |4.34817421E7           |
|Mobiles    |East  |3.8912225E7            |
|Tablets    |North |3.653276155E7          |
|Tablets    |West  |3.556450365E7          |
|Storage    |South |3.506146435E7          |
|Audio      |South |3.273420575E7          |
|Displays   |North |3.095262915E7          |
|Displays   |West  |2.67105287E7           |
|Accessories|South |2.66420428E7           |
|Audio      |West  |2.395121565E7          |
|Audio      |North |2.30163134E7           |
|Storage  

Part D

In [36]:
lazy_step1 = sales_df.filter(F.col("order_status") == "Completed")
print("1: filter transformation defined")
lazy_step2 = lazy_step1.select("product_id", "quantity", "unit_price")
print("2: select transformation defined")
lazy_step3 = lazy_step2.withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
print("3: withColumn transformation defined")
lazy_step4 = lazy_step3.groupBy("product_id").agg(F.sum("gross_amount").alias("total_gross"))
print("4: groupBy aggregation defined")
lazy_step5 = lazy_step4.orderBy(F.col("total_gross").desc())
print("5: orderBy transformation defined")
print("No Spark action executed by the transformations above")
lazy_step5.show(10, truncate=False)

1: filter transformation defined
2: select transformation defined
3: withColumn transformation defined
4: groupBy aggregation defined
5: orderBy transformation defined
No Spark action executed by the transformations above
+----------+-----------+
|product_id|total_gross|
+----------+-----------+
|P007      |552313096  |
|P006      |404480777  |
|P014      |217222759  |
|P008      |158555106  |
|P004      |120131933  |
|P015      |100484822  |
|P011      |66268636   |
|P012      |52079559   |
|P013      |44860522   |
|P010      |30048486   |
+----------+-----------+
only showing top 10 rows


In [37]:
filtered_lazy_df = sales_df.filter(F.col("quantity") > 5)
print("Execution plan:")
filtered_lazy_df.explain()
print("show() action:")
filtered_lazy_df.show(10, truncate=False)
print("count() action:")
print(filtered_lazy_df.count())

Execution plan:
== Physical Plan ==
*(1) Filter (isnotnull(quantity#52) AND (quantity#52 > 5))
+- FileScan csv [order_id#48,order_date#49,store_id#50,product_id#51,quantity#52,unit_price#53,discount_pct#54,payment_method#55,order_status#56,sales_channel#57] Batched: false, DataFilters: [isnotnull(quantity#52), (quantity#52 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,5)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


show() action:
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|O000

In [38]:
category_totals_df = (sales_df.join(products_df, "product_id", "inner")
    .groupBy("category")
    .agg(F.sum("quantity").alias("total_quantity"))
    .orderBy("category"))
category_totals = category_totals_df.collect()
for row in category_totals:
    print(row)
print("collect() is safer here because only one row per category is transferred to Python, not all 25,000 sales records.")

Row(category='Accessories', total_quantity=41839)
Row(category='Audio', total_quantity=17159)
Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Storage', total_quantity=17251)
Row(category='Tablets', total_quantity=7915)
collect() is safer here because only one row per category is transferred to Python, not all 25,000 sales records.


In [39]:
parquet_df = (sales_df.filter(F.col("order_status") == "Completed")
    .withColumn("discounted_revenue", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100.0)))
print("Transformations defined; writing triggers execution")
parquet_df.write.mode("overwrite").parquet("/content/completed_orders_discounted_revenue.parquet")
print("Parquet saved. write.mode(...).parquet(...) is the action.")

Transformations defined; writing triggers execution
Parquet saved. write.mode(...).parquet(...) is the action.


In [2]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv

File: lazy_execution_products (1).csv
Records: 15
+----------+-------------------+-----------+----------+----------+
|product_id|       product_name|   category|list_price|cost_price|
+----------+-------------------+-----------+----------+----------+
|      P001|     Wireless Mouse|Accessories|       799|       420|
|      P002|Mechanical Keyboard|Accessories|      2999|      1600|
|      P003|          USB-C Hub|Accessories|      1899|       920|
|      P004|    27-inch Monitor|   Displays|     16999|     11400|
|      P005|          Webcam HD|Accessories|      2499|      1400|
+----------+-------------------+-----------+----------+----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+

In [8]:
from pyspark.sql.functions import col
completed_df = df.filter(col("store_type")=="Mall")
completed_df.show()

+--------+---------+------+----------+
|store_id|     city|region|store_type|
+--------+---------+------+----------+
|     S01|Hyderabad| South|      Mall|
|     S03|  Chennai| South|      Mall|
|     S05|     Pune|  West|      Mall|
|     S06|    Delhi| North|      Mall|
+--------+---------+------+----------+



In [9]:
completed_df.cache()

DataFrame[store_id: string, city: string, region: string, store_type: string]

In [10]:
completed_df.count()

4

In [11]:
completed_df.show()

+--------+---------+------+----------+
|store_id|     city|region|store_type|
+--------+---------+------+----------+
|     S01|Hyderabad| South|      Mall|
|     S03|  Chennai| South|      Mall|
|     S05|     Pune|  West|      Mall|
|     S06|    Delhi| North|      Mall|
+--------+---------+------+----------+



In [12]:
from pyspark.storagelevel import StorageLevel
df.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[store_id: string, city: string, region: string, store_type: string]

In [13]:
df.show()

+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
|     S06|    Delhi| North|       Mall|
|     S07|   Jaipur| North|High Street|
|     S08|  Kolkata|  East|High Street|
+--------+---------+------+-----------+



In [14]:
df.count()

8

In [18]:
print("Before :" , df.is_cached)
df.unpersist(blocking = True)
print("After :", df.is_cached)

Before : True
After : False


In [25]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder.appName("LazyExecutionExercises").getOrCreate()
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = df.crossJoin(batch_df)

print("Original Records:", df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)







#################################################################################

# Create transformations
result_df = large_df.filter(
    F.col("city") == 'Hyderabad'
)
# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 8
Large Dataset Records: 320
+--------+---------+------+-----------+--------+
|store_id|     city|region| store_type|batch_id|
+--------+---------+------+-----------+--------+
|     S01|Hyderabad| South|       Mall|       0|
|     S02|Bengaluru| South|High Street|       0|
|     S03|  Chennai| South|       Mall|       0|
|     S04|   Mumbai|  West|High Street|       0|
|     S05|     Pune|  West|       Mall|       0|
+--------+---------+------+-----------+--------+
only showing top 5 rows
First Execution: 0.321 seconds
Second Execution: 0.07 seconds


In [31]:
from pyspark.sql import functions as F
partitioned_df = df.repartition(4)
# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    2|
|           1|    2|
|           2|    2|
|           3|    2|
+------------+-----+



In [33]:
from pyspark.sql import functions as F
partitioned_df = df.repartition(4,"store_type")
# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id","store_type") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----------+-----+
|partition_id| store_type|count|
+------------+-----------+-----+
|           1|       Mall|    4|
|           2|High Street|    4|
+------------+-----------+-----+



In [34]:
df2 = df.repartition(8)
df_repart = df2.repartition(3)
df_coalesce = df2.coalesce(3)
print("Repartition:",df_repart.rdd.getNumPartitions())
print("Coalesce:",df_coalesce.rdd.getNumPartitions())
print("Repartition Plan")
df_repart.explain()
print("Coalesce Plan")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
Repartition Plan
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1626]
         +- FileScan csv [store_id#153,city#154,region#155,store_type#156] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_stores (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<store_id:string,city:string,region:string,store_type:string>
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1626]
   +- FileScan csv [store_id#153,city#154,region#155,store_type#156] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_stores (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<store_id:string,city:string,region:string,store_type:string>


Coalesce Pl

In [35]:
from pyspark.sql import functions as F
partitioned_df = df.repartition(12)
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    1|
|           1|    1|
|           2|    1|
|           3|    1|
|           4|    1|
|           9|    1|
|          10|    1|
|          11|    1|
+------------+-----+



In [39]:
partitioned_df.rdd.getNumPartitions()

12

In [42]:
from pyspark.sql import functions as F
partitioned_df = df.repartition(12)
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    1|
|           1|    1|
|           2|    1|
|           3|    1|
|           4|    1|
|           9|    1|
|          10|    1|
|          11|    1|
+------------+-----+



In [44]:
from pyspark.sql import functions as F
partitioned_df = df.repartition(6)
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
distribution_df.groupBy("partition_id","store_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S03|    1|
|           1|     S06|    1|
|           2|     S05|    1|
|           3|     S04|    1|
|           3|     S08|    1|
|           4|     S01|    1|
|           4|     S07|    1|
|           5|     S02|    1|
+------------+--------+-----+



In [54]:
from pyspark.sql import functions as F
df2=df.repartition(12)
df_coalesce = df2.coalesce(4)
df_coalesce.rdd.getNumPartitions()

4